# Day 2 — Build (RAG + Agent) → Deploy → Monitor → Demo

---

**Last class of the bootcamp.** In 75 minutes we walk the remaining steps to a shipped, monitored, demo-able capstone.

Real work happens **between now and your demo date** (~1–2 weeks of ~1-2 hours/day).

The 4 remaining build phases:

1. **RAG pipeline** for your track
2. **Agent workflow layer**
3. **Deployment hardening + observability + cost caps**
4. **Demo prep, cost analysis, portfolio page**


## Phase 1 — RAG pipeline (~2-3 build sessions)

Two endpoints (auth required):
- `POST /kb/ingest` — file / URL / dataset ID → chunk + embed + store
- `POST /rag/ask` — question → cited answer

**Reuse Section 6 verbatim.** Adapt loaders per track.


### Track-specific ingestion

**Track A**: less doc-heavy. User-uploaded files, scraped URLs, your own product docs.

**Track B**: heaviest ingestion — 50-500 real docs. HR handbooks, contracts, product docs, tickets, meeting notes.

**Track C**: thin RAG — just enough context for the agent. SOPs, past decisions, business rules.


### The choices to make + log in your architecture doc

- **Vector DB**: Chroma (default) / pgvector / Pinecone
- **Embedder**: MiniLM (free) / BGE-small / OpenAI 3-small
- **Chunking**: recursive 500/50 default; smaller for tickets, bigger for legal
- **Retrieval**: pure semantic / hybrid (Section 6 Day 5) — pick hybrid for queries with product names, error codes, IDs
- **Reranker**: on if quality > latency

Log each pick and reason. Interviewers will ask.


### What "good enough for demo" looks like

- 10 real questions from your track return sensible answers
- Each answer cites ≥1 source
- Answers **stream** (Section 4 Day 6)
- Injection filter on user input (Section 6 Day 5)
- Distance threshold refusal for weak retrieval

Prepare **10 demo questions per track** now. You'll use them in the video.


## Phase 2 — Agent workflow layer (~2 build sessions)

Add an agent using **LangGraph** (Section 7 Day 3). What "agent" means per track:

- **Track A**: multi-step generation (plan → draft → critique → revise)
- **Track B**: research agent combining KB + web + reasoning
- **Track C**: the main product — automated business workflow


### Pick ONE clearly-defined job for your agent

**Bad**: "an agent that does everything the user asks"
**Good**:
- "an agent that turns a rough resume into a polished 1-pager" (A)
- "an agent that answers research questions using our KB + web search" (B)
- "an agent that classifies a new ticket, drafts a reply, and asks a human to approve" (C)


### Graph shapes per track

**Track A** — multi-step generation
```
plan → draft → critique → revise → done
                 ▲          │
                 └──────────┘ (retry, max 2)
```

**Track B** — research with sources
```
plan_queries → search_kb → search_web → synthesize → cite → done
```

**Track C** — automation with HITL
```
trigger → classify → gather_context → decide → HITL approve → act → done
```


### Non-negotiable requirements for the "agent" bullet on your resume

- Uses **≥2 tools** (or 2 distinct nodes)
- Explicit **stopping condition** (not just max_steps)
- **Budget** (max iterations OR max tokens) — Section 7 Day 5
- **≥1 conditional branch** — not a straight line
- All calls **traced in Langfuse**

Bonus: **HITL** step (especially Track C).


### Async execution — long jobs, short HTTP

Agents can take 20-60 seconds. Don't block HTTP:

```python
from fastapi import BackgroundTasks
JOBS = {}   # swap for a `jobs` table in production

@app.post("/agent/run")
def start(body, bg: BackgroundTasks, user = Depends(current_user)):
    job_id = uuid.uuid4().hex[:8]
    JOBS[job_id] = {"status": "running", "user": user}
    bg.add_task(run_agent, job_id, body.input)
    return {"job_id": job_id}

@app.get("/agent/{job_id}")
def status(job_id, user = Depends(current_user)):
    j = JOBS.get(job_id)
    if not j or j["user"] != user: raise HTTPException(404)
    return j
```


## Phase 3 — Deploy hardening + observability + cost caps (~1 build session)

Cheat sheet: **Section 9 Days 4 and 5.** Apply to your capstone.

**Wire Langfuse into every LLM call:**
```python
from langfuse import Langfuse
lf = Langfuse()

@lf.observe(name="retrieve")
def retrieve(q): ...

@lf.observe(as_type="generation", name="generate")
def generate(prompt): ...
```

Add `@lf.observe` to: `retrieve`, `rerank` (if used), `generate`, every agent node.


**Per-user usage table (Section 9 Day 5):**

Log every call with `user`, `endpoint`, `model`, `in_tokens`, `out_tokens`, `cost_usd`. Add `/usage` endpoint.

**Daily cost cap (dependency):**
```python
DAILY_CAP_USD = 0.20

def check_budget(user = Depends(current_user)):
    if todays_spend(user) > DAILY_CAP_USD:
        raise HTTPException(429, f"Daily budget hit")
    return user
```

Apply as `Depends(check_budget)` on every expensive endpoint. **Never demo without this.**

**Two alerts (minimal):**
- p95 latency > 5s → Langfuse Slack webhook
- Daily total spend > $X → cron that pings Slack

**Load-test with numbers.** Use Section 9 Day 6 helper. Record p50/p95/error rate → into your README.


## Phase 4 — Demo prep + cost analysis + portfolio page (~1 build session)

Three deliverables at the end:

1. **3-minute recorded demo video**
2. **Cost analysis section** in your README
3. **One-page portfolio artifact** you can send with every application


### The 3-minute demo script

- **0:00–0:15** — Hook. One sentence: what it is, who it's for, one impressive fact.
- **0:15–0:45** — Problem. Real, specific.
- **0:45–2:15** — Demo. One user story end-to-end: register → login → real question → real answer with citations → agent action → observability view.
- **2:15–2:45** — Under the hood. 5-second architecture flash. Say: "hybrid RAG, LangGraph agent, Langfuse tracing, Render deploy."
- **2:45–3:00** — CTA. "Live demo at [URL], code at [repo], looking for AI-eng roles at [type]."

Rehearse. Record on the third take. Publish to YouTube (unlisted) or Loom.


### Cost analysis section (README addition)

Grab the last 48h of your `/usage` data. Write:

```markdown
## Cost Analysis

Measured over 48h of real usage (12 test users, ~200 questions).

| Metric | Value |
|---|---|
| Avg tokens per question | 950 |
| Avg cost per question | $0.0009 |
| Cost per active user per month (projected) | $0.85 |
| Monthly infra (Render + Postgres + Chroma) | $14 |
| Break-even at $5/user/mo subscription | 3 users |

**Dominant cost**: LLM generation (~92% of variable cost).
**Optimization roadmap**:
1. Semantic cache (~-30% at 500 users)
2. Prompt caching on stable system prompt (~-15%)
3. Route classification to Llama-3B (~-8%)
```

**This section separates freshers from seniors.** No one writes it. When you do, you look 3 years more experienced.


### Portfolio page (one URL)

Options:
- **Notion public page** (easiest, looks great)
- **`/portfolio.html` on your app** (bonus points)
- **GitHub Pages** site
- **Your existing personal site**

Contents:
- Title + one-line tagline
- Video embed (YouTube / Loom / GIF)
- Live demo link (big button)
- GitHub repo link (big button)
- Cost analysis block
- Architecture diagram
- "What I'd do next" — 5 honest bullets

**This single URL becomes the link you paste into every job application.**


### Final quality bar checklist

Before you call it done:

- [ ] Live demo URL responds — no 500s
- [ ] Anonymous visitor can register → login → ask → see cited answer
- [ ] CI green on `main`
- [ ] Langfuse shows traces from the last 24h
- [ ] `/usage` returns realistic per-user numbers
- [ ] Daily cost cap works (verify by tripping it)
- [ ] 3-min video is under 3 minutes
- [ ] README has: hero, video, screenshots, architecture, cost analysis, stack, run-locally, limitations
- [ ] Portfolio URL points to everything above from one place


### Interview talking points — write them down now

Two paragraphs each, saved somewhere you'll re-read:

1. **Elevator pitch (60 sec)** — track, problem, what you built, one number.
2. **Deep dive on the hard part** — single hardest bug or design choice. What broke, what you tried, what worked, what you'd do differently.

Rehearse both. You'll use them in every interview until you're hired.


### Announce it

- **LinkedIn post** — "Shipped my capstone: [link + video + screenshot]. Looking for AI eng roles."
- **Twitter / X** — same with a GIF
- **Discord / Slack communities** you're in
- **Personal network** — DMs to 5 people

**Announcements convert 5–20× better than cold applications.** Do the announcement work.


## Recap — Section 11 total

- **Track chosen, architecture doc + cost model written** (Day 1)
- **Backend + auth deployed + green CI** (Day 1)
- **RAG pipeline** for your track (Day 2 build)
- **Agent workflow** with budget + tools + Langfuse (Day 2 build)
- **Full observability + per-user daily caps + load-tested** (Day 2 build)
- **3-min demo video + cost analysis + portfolio page** (Day 2 build)

## Recap — the whole bootcamp

Eleven sections, from "what's a Python function" to "I have a deployed AI product with observability and monitoring."

- **Section 1** — Python fundamentals
- **Section 2** — REST APIs with FastAPI + auth
- **Section 3** — Databases + SQLAlchemy
- **Section 4** — LLMs + prompt engineering
- **Section 5** — Embeddings + vector search
- **Section 6** — RAG pipeline
- **Section 7** — AI agents + LangGraph
- **Section 8** — Fine-tuning with LoRA
- **Section 9** — Deployment, CI/CD, observability
- **Section 10** — System design + interview prep
- **Section 11** — Your shipped capstone

Every one of those is a chapter in the AI-engineering job description you're about to answer.

**Now go get hired.**
